# 美股三日 ATR Pipeline — Colab

此 Notebook 會從 GitHub 取得最新版、安裝套件並執行驗證。GitHub 是 single source of truth。

> 不要把 Google service account JSON、API key 或 Token 寫入 Notebook 或 commit 到 GitHub。

In [ ]:
import os
import shutil
import subprocess

REPO_URL = "https://github.com/hh4832/us-weekly-atr-pipeline.git"
REPO_DIR = "/content/us-weekly-atr-pipeline"

# 純 Python 復原工作目錄；即使前一次執行已刪除 cwd 也能正常重跑
os.chdir("/content")
shutil.rmtree(REPO_DIR, ignore_errors=True)
subprocess.run(
    ["git", "clone", REPO_URL, REPO_DIR],
    cwd="/content",
    check=True,
)
os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    cwd=REPO_DIR,
    check=True,
)

In [ ]:
commit_hash = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True
).strip()
print(f"Git commit: {commit_hash}")

test_env = os.environ.copy()
test_env["PYTHONPATH"] = REPO_DIR
subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    cwd=REPO_DIR,
    env=test_env,
    check=True,
)

## 選用：連接 Google Sheet

只有需要從 Colab 執行 `init_google_sheet.py` 時才執行下一格。若 Sheet 已初始化完成，平常不必重跑。

執行後請上傳 `service_account.json`，再輸入 Google Sheet ID。憑證只存在本次 Colab 工作階段。

In [ ]:
from getpass import getpass
from google.colab import files
import os

uploaded = files.upload()
if "service_account.json" not in uploaded:
    raise FileNotFoundError("請上傳檔名為 service_account.json 的憑證")

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = f"{REPO_DIR}/service_account.json"
os.environ["GOOGLE_SERVICE_ACCOUNT_FILE"] = f"{REPO_DIR}/service_account.json"
os.environ["GOOGLE_SHEET_ID"] = getpass("GOOGLE_SHEET_ID：").strip()
print("憑證與 Sheet ID 已載入本次工作階段；不會寫入 GitHub。")

In [ ]:
# 僅在需要初始化／補齊程式專用工作表時執行
!python init_google_sheet.py

## 啟動 Streamlit

完成憑證設定後執行下一格。它會在 Colab 背景啟動 Streamlit，並顯示本次工作階段專用的可點擊網址。網址只在此 Colab 執行階段存續期間有效。

In [ ]:
import socket
import time
from IPython.display import HTML, display
from google.colab import output

# 重跑本格時先停止先前啟動的 Streamlit process
if "streamlit_process" in globals() and streamlit_process.poll() is None:
    streamlit_process.terminate()
    streamlit_process.wait(timeout=10)

streamlit_log = open("/content/streamlit.log", "w")
streamlit_env = os.environ.copy()
streamlit_process = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", "app.py",
        "--server.port=8501", "--server.headless=true",
    ],
    cwd=REPO_DIR,
    env=streamlit_env,
    stdout=streamlit_log,
    stderr=subprocess.STDOUT,
)

for _ in range(30):
    if streamlit_process.poll() is not None:
        streamlit_log.flush()
        raise RuntimeError("Streamlit 啟動失敗，請查看 /content/streamlit.log")
    with socket.socket() as sock:
        if sock.connect_ex(("127.0.0.1", 8501)) == 0:
            break
    time.sleep(1)
else:
    raise TimeoutError("Streamlit 在 30 秒內未啟動")

streamlit_url = output.eval_js("google.colab.kernel.proxyPort(8501)")
display(HTML(f'<a href="{streamlit_url}" target="_blank" style="font-size:18px">開啟 Streamlit App</a>'))
print(f"Streamlit PID: {streamlit_process.pid}")